In [1]:
#apply StandardScaler
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import numpy as np
import time
from sklearn.cluster import OPTICS
from sklearn.cluster import Birch
from sklearn.utils import resample
#load dataset
df= pd.read_csv("data/pain_dataset_200P_4hz.csv")
df

# Drop target and ID column & target column
X = df.drop(columns=["person_ID", "pain_scale"], errors="ignore")
print("Features shape :", X.shape)

Features shape : (96000, 7)


In [2]:
#Apply StandardScaler
#Now X_scaled contains all features standardized (mean = 0, std = 1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features shape (scaled version):", X_scaled.shape)

Features shape (scaled version): (96000, 7)


In [3]:
#Define Clustering Parameters
k_values = range(2, 9)  # clusters for KMeans, GMM, Agglomerative, Spectral
n_init = 10              # random initialization
dbscan_eps = [0.5, 1.0, 1.5]  # DBSCAN eps values
min_samples = 5

#Function to Compute Metrics
def compute_metrics(X_data, labels):
    sil = silhouette_score(X_data, labels)
    db = davies_bouldin_score(X_data, labels)
    ch = calinski_harabasz_score(X_data, labels)
    return sil, db, ch

In [4]:
#K-Means on Scaled Data
start_time = time.time()
kmean_scaled = []
for k in k_values:
    km = KMeans(n_clusters=k, n_init=n_init, random_state=42)
    labels = km.fit_predict(X_scaled)
    sil, db, ch = compute_metrics(X_scaled, labels)
    kmean_scaled.append({"algorithm": "KMeans", "preprocessing": "Scaled", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"K-Means runtime: {runtime:.4f} seconds")

Runtime: 1855.3091938495636 seconds
K-Means runtime: 1855.3092 seconds


In [5]:
#Gaussian Mixture (GMM)on Scaled Data
start_time = time.time()
gmm_scaled = []
for k in k_values:
    gmm = GaussianMixture(n_components=k, n_init=n_init, random_state=42)
    labels = gmm.fit_predict(X_scaled)
    sil, db, ch = compute_metrics(X_scaled, labels)
    gmm_scaled.append({"algorithm": "GMM", "preprocessing": "Scaled", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"GMM runtime: {runtime:.4f} seconds")

Runtime: 2581.838661670685 seconds
GMM runtime: 2581.8387 seconds


In [6]:
#Agglomerative Clustering on Scaled + PCA Data
df_small = df.sample(n=5000, random_state=42)
X_small = df_small.drop(columns=["person_ID", "pain_scale"], errors="ignore")
print("Features shape (raw version):", X_small.shape)
scaler = StandardScaler()
X_small_scaled = scaler.fit_transform(X_small)
print("Features shape (raw version):", X_small_scaled.shape)
start_time = time.time()
agg_scaled = []
for k in k_values:
    agg = AgglomerativeClustering(n_clusters=k, linkage='ward')
    agg.fit(X_small_scaled)
    labels = agg.labels_
    sil, db, ch = compute_metrics(X_small_scaled, labels)
    agg_scaled.append({"algorithm":"Agglomerative","preprocessing":"raw","k":k,"silhouette":sil,"davies_bouldin":db,"calinski_harabasz":ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Agglomerative runtime: {runtime:.4f} seconds")

Features shape (raw version): (5000, 7)
Features shape (raw version): (5000, 7)
Runtime: 11.800235509872437 seconds
Agglomerative runtime: 11.8002 seconds


In [7]:
#Spectral Clustering on Scaled Data
start_time = time.time()
# subsample = np.random.choice(len(X_scaled), 10000, replace=False)
# X_sub = X_scaled[subsample]
spec_scaled = []
for k in k_values:
    spec = SpectralClustering(n_clusters=k, affinity="nearest_neighbors")
    labels = spec.fit_predict(X_small_scaled)
    sil, db, ch = compute_metrics(X_small_scaled, labels)
    spec_scaled.append({"algorithm": "Spectral", "preprocessing": "Scaled", "k": k, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Spectral runtime: {runtime:.4f} seconds")    

Runtime: 48.4979453086853 seconds
Spectral runtime: 48.4979 seconds


In [17]:
import numpy as np

print("Shape:", X_scaled.shape)
print("NaN:", np.isnan(X_scaled).sum())
print("Inf:", np.isinf(X_scaled).sum())
print("Dtype:", X_scaled.dtype)
print("Max:", np.max(X_scaled))
print("Min:", np.min(X_scaled))

Shape: (96000, 7)
NaN: 0
Inf: 0
Dtype: float32
Max: 4.2852187
Min: -4.561943


In [8]:
#DBSCAN on Scaled Data
start_time = time.time()
dbscan_scaled = []
for eps in dbscan_eps:
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(X_scaled)
    
    # Remove noise points (-1)
    mask = labels != -1
    if np.sum(mask) > 1 and len(set(labels[mask])) > 1:  # silhouette requires >= 2 points
        sil, db, ch = compute_metrics(X_scaled[mask], labels[mask])
        dbscan_scaled.append({"algorithm": "DBSCAN", "preprocessing": "Scaled", "eps": eps, "silhouette": sil, "davies_bouldin": db, "calinski_harabasz": ch})

end_time = time.time()
runtime = end_time - start_time
print("Runtime:", runtime, "seconds")
print(f"Dbscan runtime: {runtime:.4f} seconds")

Runtime: 511.17805910110474 seconds
Dbscan runtime: 511.1781 seconds


In [15]:
#BIRCH on Scaled Data
start_time = time.time()
birch_scaled = []
#threshold_values = [0.2, 0.5, 1.0, 1.5]
threshold_values = [1.5, 3.0, 5.0, 10.0]
for t in threshold_values:
    #birch = Birch(n_clusters=None, threshold=t)
    birch = Birch(n_clusters=None, branching_factor=200, threshold=t)
    labels = birch.fit_predict(X_scaled)
    n_clusters = len(set(labels))
    #X_scaled = X_scaled.astype("float32")
    if 1 < n_clusters < len(X_scaled) :  #and len(set(labels[mask])) > 1
        sil, db, ch = compute_metrics(X_scaled, labels)
        birch_scaled.append({
            "algorithm": "BIRCH",
            "preprocessing": "Scaled",
            "threshold": t,
            "n_clusters": len(set(labels)),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })

end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"BIRCH runtime: {runtime:.4f} seconds")

Runtime: 203.61637949943542 seconds
BIRCH runtime: 203.6164 seconds


In [10]:
#Optics on Scaled Data
start_time = time.time()
optics_scaled = []
min_samples_values = [3, 5, 10, 20]

for m in min_samples_values:
    optics = OPTICS(min_samples=m, xi=0.05, n_jobs=-1)
    labels = optics.fit_predict(X_scaled)

    # Remove noise points (-1) if needed
    unique_labels = set(labels) - {-1}

    if len(unique_labels) > 1:
        sil, db, ch = compute_metrics(X_scaled, labels)
        optics_scaled.append({
            "algorithm": "OPTICS",
            "preprocessing": "Scaled",
            "min_samples": m,
            "xi": 0.05,
            "n_clusters": len(unique_labels),
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })
end_time = time.time()
runtime = end_time - start_time
# avg_time = np.mean(times)
print("Runtime:", runtime, "seconds")
print(f"Optics runtime: {runtime:.4f} seconds")

Runtime: 15350.18747973442 seconds
Optics runtime: 15350.1875 seconds


In [22]:
import csv


pain_results_scaled = (kmean_scaled + gmm_scaled + agg_scaled + spec_scaled + dbscan_scaled+birch_scaled + optics_scaled)

keys = ["algorithm", "preprocessing","k", "eps", "min_samples", "threshold","n_clusters","silhouette", "davies_bouldin", "calinski_harabasz"]

with open('updated_data/pain_new_data/pain_scaled.csv', 'w', newline='') as file:
#with open('updated_data/pain_data/pain_scaled.csv', 'w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=keys, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(pain_results_scaled)

In [ ]:
from sklearn.metrics import adjusted_rand_score
from sklearn.utils import resample
import numpy as np
import pandas as pd

# ARI stability analysis
n_bootstrap = 100
ari_results = []
# Collect all parameter settings from your previous results
all_configs = []

for r in kmean_scaled:
    all_configs.append(("K-Means", {"k": r["k"]}))

for r in gmm_scaled:
    all_configs.append(("GMM", {"k": r["k"]}))

for r in agg_scaled:
    all_configs.append(("Agglomerative", {"k": r["k"]}))

for r in spec_scaled:
    all_configs.append(("Spectral", {"k": r["k"]}))

for r in dbscan_scaled:
    all_configs.append(("DBSCAN", {"eps": r["eps"]}))

for r in birch_scaled:
    all_configs.append(("BIRCH", {"threshold": r["threshold"]}))

for r in optics_scaled:
    all_configs.append(("OPTICS", {"min_samples": r["min_samples"]}))
# helper function to fit a model and return labels 
def fit_and_predict(name, params, X_data):

    if name == "K-Means":
        model = KMeans(n_clusters=params["k"], n_init=n_init, random_state=42)
        labels = model.fit_predict(X_data)

    elif name == "GMM":
        model = GaussianMixture(n_components=params["k"], n_init=n_init, random_state=42)
        labels = model.fit(X_data).predict(X_data)

    elif name == "Agglomerative":
        model = AgglomerativeClustering(n_clusters=params["k"], linkage='ward')
        labels = model.fit_predict(X_data)

    elif name == "Spectral":
        model = SpectralClustering(
            n_clusters=params["k"],
            affinity='nearest_neighbors',
            n_init=n_init,
            random_state=42
        )
        labels = model.fit_predict(X_data)

    elif name == "DBSCAN":
        model = DBSCAN(eps=params["eps"], min_samples=min_samples)
        labels = model.fit_predict(X_data)

    elif name == "BIRCH":
        model = Birch(n_clusters=None, threshold=params["threshold"])
        labels = model.fit_predict(X_data)

    elif name == "OPTICS":
        model = OPTICS(min_samples=params["min_samples"], xi=0.05, n_jobs=-1)
        labels = model.fit_predict(X_data)

    else:
        return None

    return labels


# Reuse all parameter configurations from previous section
for algo_name, params in all_configs:

    # reference clustering on full data
    ref_labels = fit_and_predict(algo_name, params, X_scaled)

    if ref_labels is None:
        continue

    ari_scores = []
    rng = np.random.RandomState(42)

    for b in range(n_bootstrap):

        # bootstrap sample with indices
        indices = rng.choice(len(X_scaled), size=len(X_scaled), replace=True)
        X_boot = X_scaled[indices]

        boot_labels = fit_and_predict(algo_name, params, X_boot)

        if boot_labels is None:
            continue

        # compare only sampled observations
        ref_subset = np.array(ref_labels)[indices]

        # remove noise points for DBSCAN / OPTICS
        mask = (boot_labels != -1) & (ref_subset != -1)

        if np.sum(mask) < 2:
            continue

        ari = adjusted_rand_score(ref_subset[mask], np.array(boot_labels)[mask])
        ari_scores.append(ari)

    if len(ari_scores) > 0:
        ari_results.append({
            "algorithm": algo_name,
            **params,
            "ARI_mean": np.mean(ari_scores),
            "ARI_std": np.std(ari_scores)
        })


# Summary table
ari_df = pd.DataFrame(ari_results).round(4)

print("\n----BOOTSTRAP ARI STABILITY ----")
print(ari_df.to_string(index=False))

# Top 3 most stable by ARI
top3_ari = ari_df.nlargest(3, "ARI_mean")

print("\n----TOP 3 MOST STABLE BY ARI ----")
print(top3_ari.to_string(index=False))

In [ ]:
#the top three by stability score
ari_df["Stability Score"] = (1 - 2 * ari_df["ARI_std"]).clip(0, 1) #"eps", "min_samples", "threshold"

top_3 = (
    ari_df
    .sort_values(["Stability Score", "ARI_mean"], ascending=[False, False])
    .head(3)
    .loc[:, ["algorithm", "k","ARI_mean", "ARI_std", "Stability Score"]]
)

print(top_3.round(3).to_string(index=False))

In [ ]:
#the best three clustering results overall, sort primarily by ARI_mean
top_3 = ari_df.sort_values(
    ["ARI_mean", "Stability Score"],
    ascending=[False, False]
).head(3)

print(top_3.round(3).to_string(index=False))

In [ ]:
ari_df.to_csv("updated_data/ARI_Score/pain_scaled_ari.csv", index=False)

In [ ]:

# TOP 3 RESULTS FOR EACH ALGORITHM INDIVIDUALLY



all_algorithms = {
    "K-Means": kmean_scaled,
    "GMM": gmm_scaled,
    "Agglomerative": agg_scaled,
    "Spectral": spec_scaled,
    "DBSCAN": dbscan_scaled,
    "BIRCH": birch_scaled,
    "OPTICS": optics_scaled
}



for algorithm, results in all_algorithms.items():

    if len(results) == 0:
        continue

    result_df = pd.DataFrame(results)


    # Round all validation values to 4 decimal places

    result_df[
        [
            "silhouette",
            "davies_bouldin",
            "calinski_harabasz"
        ]
    ] = result_df[
        [
            "silhouette",
            "davies_bouldin",
            "calinski_harabasz"
        ]
    ].round(4)



    print("\n")
    print("="*60)
    print(algorithm)
    print("="*60)




    # Select parameter column


    parameter_columns = [
        "k",
        "eps",
        "threshold",
        "min_samples"
    ]


    parameter = None

    for col in parameter_columns:
        if col in result_df.columns:
            parameter = col
            break




    # TOP 3 SILHOUETTE


    print("\nTop 3 Silhouette Score (Higher is better)")

    top_sil = result_df.nlargest(
        3,
        "silhouette"
    )

    print(
        top_sil[
            [
                parameter,
                "silhouette"
            ]
        ].to_string(index=False)
    )



    # TOP 3 DAVIES-BOULDIN
   

    print("\nTop 3 Davies-Bouldin Index (Lower is better)")

    top_db = result_df.nsmallest(
        3,
        "davies_bouldin"
    )

    print(
        top_db[
            [
                parameter,
                "davies_bouldin"
            ]
        ].to_string(index=False)
    )



   
    # TOP 3 CALINSKI-HARABASZ


    print("\nTop 3 Calinski-Harabasz Index (Higher is better)")

    top_ch = result_df.nlargest(
        3,
        "calinski_harabasz"
    )

    print(
        top_ch[
            [
                parameter,
                "calinski_harabasz"
            ]
        ].to_string(index=False)
    )



K-Means

Top 3 Silhouette Score (Higher is better)
 k  silhouette
 2      0.2249
 3      0.1511
 4      0.1403

Top 3 Davies-Bouldin Index (Lower is better)
 k  davies_bouldin
 2          1.6519
 8          1.8781
 7          1.9465

Top 3 Calinski-Harabasz Index (Higher is better)
 k  calinski_harabasz
 2         32460.2862
 3         22062.7636
 4         17842.9611


GMM

Top 3 Silhouette Score (Higher is better)
 k  silhouette
 2      0.2136
 3      0.1459
 4      0.1244

Top 3 Davies-Bouldin Index (Lower is better)
 k  davies_bouldin
 2          1.7070
 3          2.1363
 4          2.1783

Top 3 Calinski-Harabasz Index (Higher is better)
 k  calinski_harabasz
 2         30250.0981
 3         20849.6496
 4         16234.4426


Agglomerative

Top 3 Silhouette Score (Higher is better)
 k  silhouette
 2      0.1846
 3      0.1089
 4      0.0732

Top 3 Davies-Bouldin Index (Lower is better)
 k  davies_bouldin
 2          1.8561
 8          2.3271
 7          2.4219

Top 3 Calinski-H

In [ ]:
#  Combine all algorithm results 
all_results = (
    kmean_scaled +
    gmm_scaled +
    agg_scaled +
    spec_scaled +
    dbscan_scaled +
    birch_scaled +
    optics_scaled
)

results_df = pd.DataFrame(all_results)

# Round metric values to 4 decimal places
metric_cols = ["silhouette", "davies_bouldin", "calinski_harabasz"]
results_df[metric_cols] = results_df[metric_cols].round(4)

# Columns that may exist depending on algorithm
possible_cols = ["algorithm", "k", "eps", "threshold", "min_samples","n_clusters"]

def available_cols(df, metric):
    cols = [c for c in possible_cols if c in df.columns]
    cols.append(metric)
    return cols

#Top 3 by Silhouette (higher is better)
top3_sil = results_df.nlargest(3, "silhouette")

print("\nTOP 3 SILHOUETTE ")
print(top3_sil[available_cols(results_df, "silhouette")].to_string(index=False))

#  Top 3 by Davies-Bouldin (lower is better) 
top3_db = results_df.nsmallest(3, "davies_bouldin")

print("\nTOP 3 DAVIES-BOULDIN ")
print(top3_db[available_cols(results_df, "davies_bouldin")].to_string(index=False))

# Top 3 by Calinski-Harabasz (higher is better)
top3_ch = results_df.nlargest(3, "calinski_harabasz")

print("\n TOP 3 CALINSKI-HARABASZ ")
print(top3_ch[available_cols(results_df, "calinski_harabasz")].to_string(index=False))

# Bottom 3 by Silhouette (lower is worse) 
bottom3_sil = results_df.nsmallest(3, "silhouette")

print("\n BOTTOM 3 SILHOUETTE ")
print(bottom3_sil[available_cols(results_df, "silhouette")].to_string(index=False))

#  Bottom 3 by Davies-Bouldin (higher is worse)
bottom3_db = results_df.nlargest(3, "davies_bouldin")

print("\nBOTTOM 3 DAVIES-BOULDIN ")
print(bottom3_db[available_cols(results_df, "davies_bouldin")].to_string(index=False))

# Bottom 3 by Calinski-Harabasz (lower is worse)
bottom3_ch = results_df.nsmallest(3, "calinski_harabasz")

print("\n BOTTOM 3 CALINSKI-HARABASZ ")
print(bottom3_ch[available_cols(results_df, "calinski_harabasz")].to_string(index=False))


===== TOP 3 SILHOUETTE =====
algorithm   k  eps  threshold  min_samples  n_clusters  silhouette
   KMeans 2.0  NaN        NaN          NaN         NaN      0.2249
 Spectral 2.0  NaN        NaN          NaN         NaN      0.2184
      GMM 2.0  NaN        NaN          NaN         NaN      0.2136

===== TOP 3 DAVIES-BOULDIN =====
algorithm   k  eps  threshold  min_samples  n_clusters  davies_bouldin
   DBSCAN NaN  1.0        NaN          NaN         NaN          0.7447
   DBSCAN NaN  0.5        NaN          NaN         NaN          1.1303
   OPTICS NaN  NaN        NaN          3.0      8436.0          1.2057

===== TOP 3 CALINSKI-HARABASZ =====
algorithm   k  eps  threshold  min_samples  n_clusters  calinski_harabasz
   KMeans 2.0  NaN        NaN          NaN         NaN         32460.2862
      GMM 2.0  NaN        NaN          NaN         NaN         30250.0981
   KMeans 3.0  NaN        NaN          NaN         NaN         22062.7636

===== BOTTOM 3 SILHOUETTE =====
algorithm   k  eps

In [ ]:
#